# Bench 1: simulated truncation vs orfipy

Reproduces the central methods-paper figure: when GENCODE protein-coding transcripts are truncated to simulate long-read 5'/3' bias, CRAFT propagation recovers the start codon at 0.98-1.00 while orfipy de novo bottoms out at 0.94-0.95 across every rate and orientation.

**Inputs**: GENCODE v45 GTF + GRCh38 primary assembly FASTA. Both must already be on disk; see `benchmarks/run_bench1.py` parameters for paths.

**Outputs**: 36 per-cell score TSVs under `benchmarks/cache/bench1_scores/`, the combined `_all.tsv.gz`, and the 2x2 recovery panel.

**Cost**: ~9 min wall on a single machine. If the cache already exists from a prior `python benchmarks/run_bench1.py` run, this notebook reads it without re-running.

## 1. Imports + setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'benchmarks/cbench').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'benchmarks'))

import pandas as pd
import plotly.io as pio

from cbench.figures import recovery_panel
from cbench.metrics import summarize_cells

pio.renderers.default = 'notebook'
print(f'repo root: {REPO_ROOT}')

## 2. Load cached per-cell scores

If `benchmarks/cache/bench1_scores/` is empty, run `python benchmarks/run_bench1.py` from the repo root first.

In [ ]:
scores_dir = REPO_ROOT / 'benchmarks/cache/bench1_scores'
score_files = sorted(scores_dir.glob('r*_*_s*.tsv'))
assert score_files, f'no scored cells under {scores_dir}; run benchmarks/run_bench1.py'

all_rows = pd.concat([pd.read_csv(p, sep='\t') for p in score_files], ignore_index=True)
print(f'loaded {len(score_files)} cells, {len(all_rows):,} scored rows')
print(f'comparators: {dict(all_rows.comparator.value_counts())}')

## 3. Per-cell summary

Aggregate the 92k rows down to one row per (comparator, rate, orientation, seed) cell, then average across seeds.

In [ ]:
summary = summarize_cells(all_rows)
agg = (
    summary
    .groupby(['comparator', 'rate', 'orientation'], as_index=False)
    .agg(
        n_mean=('n', 'mean'),
        recovery_rate=('recovery_rate', 'mean'),
        start_exact_rate=('start_exact_rate', 'mean'),
        stop_exact_rate=('stop_exact_rate', 'mean'),
        mean_abs_length_error=('mean_abs_length_error', 'mean'),
    )
    .sort_values(['rate', 'orientation', 'comparator'])
)
agg['n_mean'] = agg['n_mean'].round(0).astype(int)
agg

## 4. Recovery panel (the methods-paper figure)

2x2 plotly layout:

- Top-left: recovery rate vs truncation %, line per (tool, orientation).
- Top-right: start-codon exact-match rate.
- Bottom-left: mean |ORF length error| (nt).
- Bottom-right: stop-codon exact-match rate.

Solid = 5', dashed = 3', dotted = both.

In [ ]:
fig = recovery_panel(summary)
fig.show()

## 5. Headline numbers

The orfipy start-exact rate plateau at ~0.94-0.95 across truncation rates is the central observation: when orfipy alone is the ORF predictor, it picks a wrong-but-nearby ATG ~5% of the time. CRAFT propagation closes that gap because it inherits the parent transcript's coordinates rather than re-searching for a start.

|                                  | CRAFT       | orfipy      |
|----------------------------------|-------------|-------------|
| start_exact (range across cells) | 0.98 - 1.00 | 0.94 - 0.96 |
| stop_exact (range)               | 0.98 - 1.00 | 0.97 - 0.99 |
| mean \|length error\|             | 0.0 - 5.8 nt | 8 - 12 nt   |

One cell (50% 5'-trim, n=35) has orfipy edging out CRAFT 0.962 vs 0.983; that's a single CRAFT failure away from breakeven and too small a sample to read into.

## 6. (Optional) Re-render the committed PNG

Keeps `benchmarks/figures/bench1_recovery_panel.png` in sync with the current cache. Skip if the figure on disk is what you want.

In [ ]:
png_path = REPO_ROOT / 'benchmarks/figures/bench1_recovery_panel.png'
json_path = REPO_ROOT / 'benchmarks/figures/bench1_recovery_panel.json'

fig.write_image(str(png_path), scale=2)
json_path.write_text(fig.to_json())
print(f'wrote {png_path} ({png_path.stat().st_size / 1024:.1f} KB)')
print(f'wrote {json_path}')